In [ ]:
!pip install alpha_vantage

In [ ]:
import pandas as pd
import requests
import json
from alpha_vantage.timeseries import TimeSeries

api_key = 'DGLLCB818CWYTWLU'
symbol = 'AAPL'
url = f'https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol={symbol}&interval=5min&apikey={api_key}'
r = requests.get(url)

In [ ]:
data = r.json()['Daily Time Series']

In [ ]:
# Importando os dados temporais mensais para um Data Frame
df = data
df = pd.DataFrame(data).T.reset_index()

# Renomiando as colunas e organizando os tipos de dados das colunas
df = pd.DataFrame.from_dict(data, orient='index').reset_index()
df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
df = df.astype({'Date': 'datetime64[ns]', 'Open': 'float', 'High': 'float', 'Low':'float', 'Close': 'float', 'Volume': 'float'})

In [ ]:
# Lista de colunas numéricas
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']

# Aplicando o método IQR em todas as colunas
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Mantendo apenas os valores dentro dos limites
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

In [ ]:
# Normalização dos dados
cols_to_normalize = ['Open', 'High', 'Low', 'Close', 'Volume']
df[cols_to_normalize] = (df[cols_to_normalize] - df[cols_to_normalize].mean()) / df[cols_to_normalize].std()

In [ ]:
#Exportar em .parquet
df.to_parquet('timeseries_monthly_APPLE.parquet', engine='pyarrow')